In [1]:
# Import libraries
import os
from os import listdir
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Convert df to gdf
# https://stackoverflow.com/questions/71907567/valueerror-geodataframe-does-not-support-multiple-columns-using-the-geometry-co
# Coordinate system of osm: http://download.geofabrik.de/osm-data-in-gis-formats-free.pdf
# must have a geometry column in csv
def csv_2_gdf(df):
    gdf = gpd.GeoDataFrame(df.loc[:, [c for c in df.columns if c != "geometry"]], 
                           geometry=gpd.GeoSeries.from_wkt(df["geometry"]),
                           crs="epsg:4326",)
    return gdf

### All states except California, Florida, Texas

In [3]:
# file_path = r"D:\Work\Box Sync\TRB 2024\Extracted_links" 
file_path = r"D:\Work\Box Sync\TRB 2024\Extracted_links\Temporary"
os.chdir(file_path)

# Reading and saving the links data as a dict
links = {}
# iterate through all file
for file in os.listdir():
    # Check whether file is in text format or not
    if file.endswith("link.csv"):
        links[file[:-4]] = pd.read_csv(file, encoding='cp1252')

In [4]:
# # NOT USING THE NODES
# # Reading and saving the nodes data as a dict
# nodes = {}
# # iterate through all file
# for file in os.listdir():
#     # Check whether file is in text format or not
#     if file.endswith("node.csv"):
#         nodes[file[:-4]] = pd.read_csv(file, encoding='cp1252', low_memory=False)

In [5]:
# print(links['utah_link'].shape)
# # Check if there's multiple links for a node pair
# if links['utah_link'].groupby(['from_node_id', 'to_node_id']).size().sum() == links['utah_link'].shape[0]:
#     print('===One link connecting a node pairs===')

In [6]:
print(links.keys())

dict_keys(['alabama_link', 'arizona_link', 'arkansas_link', 'colorado_link', 'connecticut_link', 'delaware_link', 'district-of-columbia_link', 'georgia_link', 'hawaii_link', 'idaho_link', 'illinois_link', 'indiana_link', 'iowa_link', 'kansas_link', 'kentucky_link', 'louisiana_link', 'maine_link', 'maryland_link', 'massachusetts_link', 'michigan_link', 'minnesota_link', 'mississippi_link', 'missouri_link'])


In [7]:
# Exporting all shapefiles for plotting
# Importing the places shapefiles for all states
# combining them into one for further use
from pathlib import Path
# define the file location
folder = Path(r"D:\Work\Box Sync\TRB 2024\US_places_2020\temp\\")
# reading the zip file
shapefiles = folder.glob("tl_2020_*_place.zip")
# combining places for US into one file
gdf = [gpd.read_file(shp) for shp in shapefiles]


In [8]:
len(gdf), type(gdf)

(23, list)

In [9]:
# MAKE SURE THAT THESE TWO DICTS ARE ORDERED CORRECTLY TO MAKE THE LOOP WORK
state_code = ['01','04','05', '08','09','10','11','13','15','16','17','18','19','20','21','22','23','24','25', 
              '26','27','28','29','30', '31','32','33','34','35','36','37','38','39','40','41','42',
               '44','45','46','47','49','50','51','53','54','55','56']

# state_name = links.keys()
# state_dict = dict(zip(state_name, state_code))
print(len(state_code))

state_names_gdf = ['alabama_link', 'arizona_link', 'arkansas_link', 'colorado_link', 'connecticut_link', 'delaware_link',
                   'district-of-columbia_link', 'georgia_link', 'hawaii_link', 'idaho_link', 'illinois_link', 'indiana_link',
                   'iowa_link', 'kansas_link', 'kentucky_link', 'louisiana_link', 'maine_link', 'maryland_link', 'massachusetts_link',
                   'michigan_link', 'minnesota_link', 'mississippi_link', 'missouri_link', 'montana_link', 'nebraska_link', 
                   'nevada_link', 'new-hampshire_link', 'new-jersey_link', 'new-mexico_link', 'new-york_link', 'north-carolina_link',
                   'north-dakota_link', 'ohio_link', 'oklahoma_link', 'oregon_link', 'pennsylvania_link', 
                   'rhode-island_link', 'south-carolina_link', 'south-dakota_link', 'tennessee_link', 'utah_link', 'vermont_link', 
                   'virginia_link', 'washington_link', 'west-virginia_link', 'wisconsin_link', 'wyoming_link']
                   
print(len(state_names_gdf))
place_dict = dict(zip(state_names_gdf, gdf))

23
23


In [10]:
# CHECK
if len(links.keys()) != len(gdf):
    print("!!! Lengths do not MATCH !!! CHECK WHETHER SHAPEFILES ALIGN WITH STATE CODES")
else:
    print("Go ahead..")

Go ahead..


In [11]:
import shapely
import geopandas as gpd
from pyproj import CRS
import utm

# assign utm based on the lat long value of a geometry
def utm_crs_from_latlon(lat, lon):
    crs_params = dict(
        proj = 'utm',
        zone = utm.latlon_to_zone_number(lat, lon),
        south = lat < 0
        )
    return CRS.from_dict(crs_params)

# Get projected geometry to create buffer for place polygons
def get_utm_projected(geom_wkt): # geom is shapely geometry file
    geom_shape = shapely.wkt.loads(geom_wkt)
    utm_crs = utm_crs_from_latlon(lat = geom_shape.centroid.y, lon = geom_shape.centroid.x)
    # print(utm_crs)
    line_geo = gpd.GeoSeries([geom_shape], crs='EPSG:4326')
    # taking a buffer of 3.6 meters to include th roads along the periphery
    return line_geo.to_crs(utm_crs).buffer(3.6)

# reads link files from a list of link dataframes for each place
def read_osm_links(key):
    state_streets = csv_2_gdf(links[key]) # 'alabama_link'
    state_streets_major = state_streets[(state_streets['link_type'] < 7) | (state_streets['link_type'] == 15)]
    # Create a new column with sorted tuples
    state_streets_major['sorted_nodes'] = list(zip(state_streets_major['from_node_id'], state_streets_major['to_node_id']))
    state_streets_major['sorted_nodes'] = state_streets_major['sorted_nodes'].apply(lambda x: tuple(sorted(x)))
    return state_streets_major

In [12]:
geoids = []
name_of_places = []
clipped_to_places = []

# clipping roads at state level to places 
for key in links.keys():
    print(key)
    state_streets_major = read_osm_links(key)      
    state_places = place_dict[key]

    geoids.append(state_places.GEOID)
    name_of_places.append(state_places.NAMELSAD)
    gdf_wkt = state_places.geometry.to_wkt()
    
    # Clip link gdf by multipolygons
    # Creates a list of clipped files per place in states_places
    clipped_streets = [gpd.clip(state_streets_major, get_utm_projected(geom_wkt).to_crs(state_streets_major.crs)) for geom_wkt in gdf_wkt]
    clipped_to_places.append(clipped_streets)

print(len(clipped_to_places))

alabama_link


c:\Users\uttar\miniforge3\envs\analyzeInfra\Lib\site-packages\geopandas\geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


arizona_link


c:\Users\uttar\miniforge3\envs\analyzeInfra\Lib\site-packages\geopandas\geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


arkansas_link


...

mississippi_link


c:\Users\uttar\miniforge3\envs\analyzeInfra\Lib\site-packages\geopandas\geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


missouri_link


c:\Users\uttar\miniforge3\envs\analyzeInfra\Lib\site-packages\geopandas\geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


23


In [13]:
# pd.set_option('display.max_colwidth', None)

In [14]:
# clipped_x = clipped_to_places[0][58]
# clipped_y = clipped_x[~clipped_x.is_empty]
# gdf_wkt = clipped_y.geometry.to_wkt()
# for geom in gdf_wkt:
#     # # Convert to accurate UTM: https://gis.stackexchange.com/questions/436938/calculate-length-using-geopandas
#     # print(geom)
#     # read geometry as a shapely geometry
#     line_shape = shapely.wkt.loads(geom)
#     if line_shape.is_empty:
#         print('empty geom')
#     else:
#         # indentify the centroid of the geometry
#         utm_crs = utm_crs_from_latlon(lat = line_shape.centroid.y,
#                                     lon = line_shape.centroid.x)       

#         line_geo = gpd.GeoSeries([line_shape], crs='EPSG:4326')
#         print(line_geo.to_crs('EPSG:3857').length.values)
#         print(line_geo.to_crs(utm_crs).length.values)
#         count = count+1
#         print(count)

In [15]:
no_roads = []
name_no_roads = []

for i, state_clipped in enumerate(clipped_to_places):
    clipped_projected_lengths = []
    for j, clipped in enumerate(state_clipped):
        # print(j)
        clipped = clipped[~clipped.is_empty]
        if clipped.shape[0] != 0:
            print(f'Shape of the roadway clipped by place at index {j} for state {i} is {clipped.shape}')
            output_df = pd.DataFrame(clipped)
            utm_length = []
            crs_length = []
            # convert geodataframe geometry to wkt format
            gdf_wkt = clipped.geometry.to_wkt()
            for geom in gdf_wkt:
                # # Convert to accurate UTM: https://gis.stackexchange.com/questions/436938/calculate-length-using-geopandas
                # read geometry as a shapely geometry
                line_shape = shapely.wkt.loads(geom)
                # indentify the centroid of the geometry
                utm_crs = utm_crs_from_latlon(lat = line_shape.centroid.y,
                                              lon = line_shape.centroid.x)       

                line_geo = gpd.GeoSeries([line_shape], crs='EPSG:4326')
                crs_length.append(line_geo.to_crs('EPSG:3857').length.values)
                utm_length.append(line_geo.to_crs(utm_crs).length.values)

            if output_df.shape[0] == len(utm_length):
                try:
                    output_df['utm_length'] = utm_length
                    output_df['crs_length'] = crs_length
                except Exception as e:        
                    print(f"!!!====== ERROR ======!!! :\n  {file}")
                    print(f"!!!File shapes do not match!!\n")
                    continue

            clipped_projected_lengths.append(output_df)

        else:
            print('Places and geoids with missing geometry, so no x y values:====')
            print(geoids[i][j], name_of_places[i][j])
            no_roads.append(geoids[i][j])
            name_no_roads.append(name_of_places[i][j])
            # remove element at index i from the geoid list
            geoids[i].pop(j)
            name_of_places[i].pop(j)
    
    print(len(clipped_projected_lengths))

    error_opening = []

    motorway_roads = []
    trunk_roads = []
    primary_roads = []
    secondary_roads = []
    tertiary_roads = []
    unclassified_roads = []
    residential_roads = []

    cl_motorway_roads = []
    cl_trunk_roads = []
    cl_primary_roads = []
    cl_secondary_roads = []
    cl_tertiary_roads = []
    cl_unclassified_roads = []
    cl_residential_roads = []

    for k, clipped_links in enumerate(clipped_projected_lengths):
        try:                
            # Filter the data and calculate the sum of road lengths for each road type               
            road_lengths = clipped_links.groupby('link_type_name')['utm_length'].sum()
            cl_road_lengths = clipped_links.groupby(['link_type_name', 'sorted_nodes']).agg({'utm_length':'mean'}).groupby('link_type_name').sum()
            cl_road_lengths = cl_road_lengths.squeeze(axis=1)
            
            # Append the calculated road lengths to the respective lists
            motorway_roads.append(road_lengths.get('motorway', 0))
            trunk_roads.append(road_lengths.get('trunk', 0))
            primary_roads.append(road_lengths.get('primary', 0))
            secondary_roads.append(road_lengths.get('secondary', 0))
            tertiary_roads.append(road_lengths.get('tertiary', 0))
            unclassified_roads.append(road_lengths.get('unclassified', 0))
            residential_roads.append(road_lengths.get('residential', 0))

            cl_motorway_roads.append(cl_road_lengths.get('motorway', 0))
            cl_trunk_roads.append(cl_road_lengths.get('trunk', 0))
            cl_primary_roads.append(cl_road_lengths.get('primary', 0))
            cl_secondary_roads.append(cl_road_lengths.get('secondary', 0))
            cl_tertiary_roads.append(cl_road_lengths.get('tertiary', 0))
            cl_unclassified_roads.append(cl_road_lengths.get('unclassified', 0))
            cl_residential_roads.append(cl_road_lengths.get('residential', 0))
            # print("finished upto ", str(k))
               
        except Exception as e:
            error_opening.append(clipped_links.index)
            print(f"Error opening file clipped_links at position {k} of clipped_projected_lengths: {e}")

    list_of_tuples = list(zip(geoids[i], name_of_places[i], motorway_roads, trunk_roads, primary_roads, secondary_roads, 
                          tertiary_roads, unclassified_roads, residential_roads, cl_motorway_roads, cl_trunk_roads, cl_primary_roads,
                          cl_secondary_roads, cl_tertiary_roads, cl_unclassified_roads, cl_residential_roads))
    
    df_roads = pd.DataFrame(list_of_tuples, columns = ['GEOID', 'NAMELSAD','motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential',
                                                    'cl_motorway', 'cl_trunk', 'cl_primary', 'cl_secondary', 'cl_tertiary', 'cl_unclassified', 'cl_residential'])

    print(df_roads.shape)
    df_roads.to_csv(r'D:\Work\Box Sync\Quantify Infrastructure\Streets_df\All states\df_roads_utm_'+ str(i) + '.csv')

Shape of the roadway clipped by place at index 0 for state 0 is (883, 19)
Shape of the roadway clipped by place at index 1 for state 0 is (64, 19)
Shape of the roadway clipped by place at index 2 for state 0 is (332, 19)
Shape of the roadway clipped by place at index 3 for state 0 is (452, 19)
Shape of the roadway clipped by place at index 4 for state 0 is (196, 19)
Shape of the roadway clipped by place at index 5 for state 0 is (649, 19)
Shape of the roadway clipped by place at index 6 for state 0 is (1703, 19)
Shape of the roadway clipped by place at index 7 for state 0 is (69, 19)
Shape of the roadway clipped by place at index 8 for state 0 is (985, 19)
Shape of the roadway clipped by place at index 9 for state 0 is (478, 19)
Shape of the roadway clipped by place at index 10 for state 0 is (10633, 19)
Shape of the roadway clipped by place at index 11 for state 0 is (3163, 19)
Shape of the roadway clipped by place at index 12 for state 0 is (2624, 19)
Shape of the roadway clipped by 

In [16]:
missing_places = {no_roads[i]: name_no_roads[i] for i in range(len(no_roads))}
print(len(missing_places))
missing_places

2


{'0405450': 'Bear Flat CDP', '0812030': 'Carbonate town'}

In [ ]:
# clipped_projected_lengths[1]
# clipped_to_places[1]